# 01 — ROI-to-anatomy geometry review

Review where each functional ROI falls on the anatomy image. Genes and response measures are not used in this review.

In [ ]:
# 02 — Choose the fish and load the review tools
from pathlib import Path
import pandas as pd
from codeants_2pf_hcr.qc_notebooks import QCNotebookConfig, resolve_qc_notebook_context, write_qc_review_record
from codeants_2pf_hcr.spatial import load_or_cache_voxels
import importlib
import codeants_2pf_hcr.plots.qc_geometry as qc_geometry
# Rerun this setup cell after editing qc_geometry; restart for context/schema changes.
qc_geometry = importlib.reload(qc_geometry)
from codeants_2pf_hcr.plots.qc_geometry import load_geometry_review_bundle, render_geometry_placement_overview, show_geometry_review_dashboard, summarize_geometry_review

FISH_ID = 'L765_f04'
LOCAL_ROOT = Path('/Volumes/dataDrive/dataProcessing/2p_processing')
PIPELINE_ROOT = Path('/Volumes/dataDrive/dataProcessing/2p_processing/_staged_pipeline_runs/20260915T093051Z_WORKING_PIPELINE_AND_QC/L765_f04')


In [ ]:
# 03 — Load the geometry results
qc_context = resolve_qc_notebook_context(QCNotebookConfig(
    fish_id=FISH_ID,
    local_root=LOCAL_ROOT,
    pipeline_root=PIPELINE_ROOT,
))
geometry_root = qc_context.stage_paths['match-roi-to-anatomy'] / 'registration'
geometry = load_geometry_review_bundle(
    geometry_root,
    expected_fish_id=FISH_ID,
    anatomy_stack_path=qc_context.fish_dir / '02_reg' / '00_preprocessing' / '2p_anatomy' / f'{FISH_ID}_anatomy_2P_GCaMP.nrrd',
    anatomy_labels_path=qc_context.fish_dir / '03_analysis' / 'structural' / 'cp_masks' / f'{FISH_ID}_anatomy_00001_uint8_8bit_cp_masks.tif',
    plane_refs_path=geometry_root / 'plane_refs_summary_geometry.json',
    transformed_roi_root=qc_context.stage_paths['transform-functional-rois-to-anatomy'],
)
display(summarize_geometry_review(geometry.matches))

In [ ]:
# 04 — Display the geometry summary
geometry_dashboard = show_geometry_review_dashboard(geometry, max_rows=50)
display(geometry_dashboard['figure'])

## 05 — Review in-plane label placement

Each panel uses the best anatomy Z for one functional plane. Green outlines are anatomy labels and magenta outlines are transformed functional ROIs. Matching outlines on the same anatomy features support accurate in-plane placement.

In [ ]:
# 06 — Display anatomy, anatomy-label, and functional-ROI overlays
PLACEMENT_OVERVIEW = render_geometry_placement_overview(geometry)
display(PLACEMENT_OVERVIEW)

## 07 — Review centroid offsets in anatomy context

Choose a plane for detailed review. The anatomy image and both label outlines show where each centroid link sits. The violin plot compares every plane's XY offsets with the median anatomy-label radius.

In [ ]:
# 08 — Display centroid offsets with anatomy context
PLANE_INDEX = 9
CENTROID_OFFSET_FIGURE = qc_geometry.render_centroid_offset_detail(geometry, plane_idx=PLANE_INDEX)
display(CENTROID_OFFSET_FIGURE)

## 09 — Q0.1 legacy `[53a]` versus Notebook 03 parity audit

This is read-only: it applies legacy `[53a]`'s anatomy-match plus unique-match filter to the legacy ROI master and to frozen Notebook 03 geometry, intersects exact `(plane_idx, func_label)` keys, then compares legacy `selected_dist_um` with a physical centroid remeasurement. Review the displayed anatomy voxel spacing, anatomy-space coordinate columns, and selected functional-to-anatomy transform direction before accepting. Do not redesign geometry panels until this audit is reviewed.

In [ ]:
if geometry.anatomy_stack_path is None: raise RuntimeError("Configure the prepared anatomy stack before running the parity audit.")
_vox = load_or_cache_voxels(geometry.anatomy_stack_path)
ANATOMY_XY_SPACING_UM = (float(_vox["X"]), float(_vox["Y"]))  # prepared anatomy NRRD header
STORED_DISTANCE_CANDIDATES = (
    qc_context.stage_paths["assign-hcr-identity"] / "registration" / "functional_roi_activity_identity.csv",
    qc_context.stage_paths["score-activity-bpi"] / "registration" / "functional_roi_activity_identity.csv",
)
STORED_DISTANCE_ROI_TABLE = next((path for path in STORED_DISTANCE_CANDIDATES if path.is_file()), None)
if STORED_DISTANCE_ROI_TABLE is None: raise FileNotFoundError("Stored-distance audit requires a frozen staged ROI master; checked: " + ", ".join(map(str, STORED_DISTANCE_CANDIDATES)))
stored_distance_rows = pd.read_csv(STORED_DISTANCE_ROI_TABLE)
XY_OFFSET_AUDIT = qc_geometry.build_stored_vs_physical_xy_offset_audit(stored_distance_rows, geometry.matches, dx_um=ANATOMY_XY_SPACING_UM[0], dy_um=ANATOMY_XY_SPACING_UM[1])
XY_OFFSET_AUDIT_SUMMARY = qc_geometry.summarize_stored_vs_physical_xy_offset_audit(XY_OFFSET_AUDIT)
XY_OFFSET_AUDIT_FIGURE = qc_geometry.render_stored_vs_physical_xy_offset_audit(XY_OFFSET_AUDIT, fish_id=FISH_ID)
TRANSFORM_DIRECTION_AUDIT = "functional ROI centroid → anatomy space; no additional transform in Notebook 03"
display({"stored_distance_source": str(STORED_DISTANCE_ROI_TABLE), "anatomy_xy_spacing_um": ANATOMY_XY_SPACING_UM, "transform_direction": TRANSFORM_DIRECTION_AUDIT})
display(XY_OFFSET_AUDIT_SUMMARY)
display(XY_OFFSET_AUDIT_FIGURE)
display(XY_OFFSET_AUDIT[["plane_idx", "func_label", "stored_centroid_x_func_anat", "stored_centroid_y_func_anat", "stored_centroid_x_anat", "stored_centroid_y_anat", "qc_centroid_x_func_anat", "qc_centroid_y_func_anat", "qc_centroid_x_anat", "qc_centroid_y_anat", "stored_xy_offset_value", "physical_xy_offset_um"]].head())

## 09 — Record the review decision

Saving records what you reviewed. It does not change the geometry. Check unmatched ROIs and ROIs that lost a one-to-one assignment before accepting the result.

In [ ]:
# 10 — Set the optional review note
SAVE_REVIEW = False  # Set True only after an explicit human review decision.
REVIEWER = 'Danin'
DECISION = 'review_required'  # accepted, rejected, or review_required
NOTES = ''

if SAVE_REVIEW:
    review_path = write_qc_review_record(
        qc_context,
        stage_name='match-roi-to-anatomy',
        decision=DECISION,
        reviewer=REVIEWER,
        reviewed_artifacts=[geometry.matches_path, geometry.summary_path, geometry.plane_meta_path],
        notes=NOTES,
    )
    print(review_path)